In [ ]:
from fastai.text.all import *
from fastai.text.tokenizer import *

/Users/hoa/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Data

`tokenize_dataset` downloads the dataset, encodes every article with GPT-2 and saves the token cache on the first call. Later calls load the cache. `split_docs` gives one array of token ids per article.

In [ ]:
flat, offsets, tok = tokenize_dataset('answerdotai/simplewiki', 'md', 'articles')
docs = split_docs(flat, offsets)
len(docs), len(flat)

(270509, 183842357)

In [ ]:
tok.decode(docs[0][:50])

'**April** (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.\n\nApril always begins on the same day of the'

## Datasets

`HfTokenizer` is the one transform of the pipeline. It adds `<bos>` and `<eos>` around each article and decodes a batch back to text. `Datasets` with `dl_type=LMDataLoader` is what `TextBlock(is_lm=True)` builds in the tutorial.

In [ ]:
tokenizer = HfTokenizer(tok)
splits = RandomSplitter(valid_pct=0.1, seed=42)(docs)
dsets = Datasets(docs, [tokenizer], splits=splits, dl_type=LMDataLoader)

In [ ]:
bs,sl = 8,72
dls = dsets.dataloaders(bs=bs, seq_len=sl)
dls.show_batch()

<bos>The **New Hampshire Senate** is the upper house of the New Hampshire General Court. The Senate has been meeting since 1784. The Senate is made of 24 members representing Senate districts.

<eos><bos>**Northridge** is a neighborhood of Los Angeles, California in the San Fernando Valley. It is the home of California State University, Northridge.


|-id\=307 bgcolor\=\#fefefe
\| 136307 \|\| <template data-op="mediawiki:transclude" data-name="mp"><div data-arg>2004 BX</div><div data-arg>43</div></template> \|\| — \|
 of the Order of the British Empire (OBE) in the 2017 New Year Honours for services to drama.

Pigott-Smith was born in Rugby, Warwickshire. He studied at Bristol University. In 1972, he married Pamela Miles. Pigott-Smith died on 7 April 2017 in Northampton, Northamptonshire
 of the Chicago Blues". A lot of musicians who played in his band later were stars of the blues scene themself. (Otis Spann, Little Walter, Junior Wells, Big Walter Horton, James Cotton...). Willie Dix

## Model



In [ ]:
config = awd_lstm_lm_config.copy()
config.update({'input_p': 0.6, 'output_p': 0.4, 'weight_p': 0.5, 'embed_p': 0.1, 'hidden_p': 0.2})
model = get_language_model(AWD_LSTM, len(dls.vocab), config=config)

In [ ]:
opt_func = partial(Adam, wd=0.1, eps=1e-7)
cbs = [MixedPrecision(), GradientClip(0.1)] + rnn_cbs(alpha=2, beta=1)
learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), opt_func=opt_func, cbs=cbs, metrics=[accuracy, Perplexity()])

## Training

In [ ]:
learn.fit_one_cycle(1, 5e-3, moms=(0.8,0.7,0.8), div=10)